# aa2codon v2: PyTorch training
Select a GPU runtime in Colab (Runtime > Change runtime type). Train on one host or strain per ensemble. Legacy TensorFlow models are not loaded.

In [ ]:
!git clone -b pytorch-xielu-uncertainty https://github.com/pyypyyy/aa2codon.git /content/aa2codon_v2
%cd /content/aa2codon_v2/pytorch_v2
!pip -q install biopython captum plotly tensorboard tqdm
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
print(*[str(p) for p in Path('../Genomes').rglob('*') if p.is_file()], sep='\n')

In [ ]:
# Specify CDS FASTA paths for ONE intended host.
FASTA = [] # e.g. ['/content/drive/MyDrive/my_host_cds.fasta']
OUT = '/content/drive/MyDrive/aa2codon_v2/models'
TUNED = '/content/drive/MyDrive/aa2codon_v2/tuning/best_config.json'
USE_TUNED = True # Set False to use the baseline without tuning.
ENSEMBLE = 10
LIVE_MONITOR = True # Set False for text output; logs are still saved.
EPOCHS = 40
from train import run
from model import Config
import json
assert FASTA, 'Set FASTA to your chosen CDS file(s).'
if USE_TUNED:
    from tune import signature
    tuned = json.loads(Path(TUNED).read_text())
    sig = tuned['signature']
    current = signature(FASTA, sig['batch'], sig['max_length'], sig['epochs'], sig['seed'], sig['prune'], None)
    if current != sig:
        raise ValueError('Tuning dataset/settings do not match. Use the same FASTA files or retune.')
    config = Config(**tuned['config'])
    batch, lr, wd = tuned['batch'], tuned['lr'], tuned['weight_decay']
else:
    config = Config(d_model=256, heads=8, layers=3, d_ff=1024, dropout=.05, activation='xielu', max_length=1024)
    batch, lr, wd = 16, 3e-4, .01
manifest, report = run(FASTA, OUT, config, ensemble=ENSEMBLE, batch=batch, epochs=EPOCHS, lr=lr, weight_decay=wd, live=LIVE_MONITOR)

In [ ]:
from visualization import calibration_plot
calibration_plot(report).show()

## TensorBoard (optional)
Run this cell after training or in another notebook connected to the same Drive. The live Plotly monitor above is updated every epoch.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUT}/training_logs